# **National Parks Biodiversity: Data Cleaning and Preparation**

## **1. Project overview**
This project analyzes species observations recorded across four U.S. national parks. Before any analysis is possible, the two raw source files have been inspected, cleaned, and combined into a single, trustworthy table. This notebook covers that preparation stage.

- The approach to the analysis of this data set was in the perspective of a biodiversity analyst working with the National Park Service.
- Supporting the National Park Service’s mission to protect at-risk species while maintaining biodiversity across parks.
- Identify key characteristics of species, examine their conservation status, and explore how these species are connected to the parks in which they are found.

### **Datasets**
| File | Rows | Columns | Description |
|---|---|---|---|
| `observations.csv` | 23,296 | 3 | Observation counts per species per park (`scientific_name`, `park_name`, `observations`) |
| `species_info.csv` | 5,824 | 4 | Species metadata (`category`, `scientific_name`, `common_names`, `conservation_status`) |

## **2. Data Loading and Setup**

This section loads the two raw datasets and prepares them for cleaning:
- **`observations.csv`**: observation counts per species in each national park
- **`species_info.csv`**: species metadata, including category and conservation status

File paths are built with `pathlib.Path` so the notebook runs on any operating system. Each raw file is loaded once into `df_raw_1` and `df_raw_2`, and I then create working copies (`df_observations`, `df_species`). All cleaning happens on the copies, so the raw data stays untouched and any step can be traced back to the original.

In [555]:
import pandas as pd
from pathlib import Path

# Locate the raw observation and species datasets.
observations_path = Path("../data/raw/observations.csv")
species_path = Path("../data/raw/species_info.csv")

# Load the raw datasets, keeping them unchanged for reference.
df_raw_observations = pd.read_csv(observations_path)
df_raw_species = pd.read_csv(species_path)

# Create working copies for cleaning and preparation.
df_observations = df_raw_observations.copy()
df_species = df_raw_species.copy()

## **3. Data Inspection**

Both datasets where inspected to understand their structure, column types, and contents. This section previews the first rows, summarizes each column, and checks the shape and data types.

### **3.1 Data Preview**
Started with `df_observations`, which records how many times each species was observed in each park.

In [556]:
# Preview the first five rows of the observations dataset.
df_observations.head()

,scientific_name,park_name,observations
0,Vicia benghalensis,Great Smoky Mountains National Park,68
1,Neovison vison,Great Smoky Mountains National Park,77
2,Prunus subcordata,Yosemite National Park,138
3,Abutilon theophrasti,Bryce National Park,84
4,Githopsis specularioides,Great Smoky Mountains National Park,85


Each row is one species in one park, with its observation count. The three columns are:

- `scientific_name`: the species identifier 
- `park_name`: the national park where the observations were recorded
- `observations`: the number of observations (numeric)

Next, preview of `df_species`, which holds the metadata for each species.

In [557]:
# Preview the first five rows of the species dataset.
df_species.head(10)

,category,scientific_name,common_names,conservation_status
0,Mammal,Clethrionomys gapperi gapperi,Gapper's Red-Backed Vole,NaN
1,Mammal,Bos bison,"American Bison, Bison",NaN
2,Mammal,Bos taurus,"Aurochs, Aurochs, Domestic Cattle (Feral), Dom...",NaN
3,Mammal,Ovis aries,"Domestic Sheep, Mouflon, Red Sheep, Sheep (Feral)",NaN
4,Mammal,Cervus elaphus,Wapiti Or Elk,NaN
5,Mammal,Odocoileus virginianus,White-Tailed Deer,NaN
6,Mammal,Sus scrofa,"Feral Hog, Wild Pig",NaN
7,Mammal,Canis latrans,Coyote,Species of Concern
8,Mammal,Canis lupus,Gray Wolf,Endangered
9,Mammal,Canis rufus,Red Wolf,Endangered


- Each row describes one species, with its `category`, `scientific_name`, `common_names`, and `conservation_status`
- All 10 rows are in the Mammal category
- `common_names` stores one or more names per species in a single comma-separated string. Row 2 lists "Aurochs" twice, and its value is cut off in the display
- `conservation_status` is missing (`NaN`) for 7 of the 10 rows
- The three rows with a status are *Canis latrans* (Species of Concern), *Canis lupus* (Endangered), and *Canis rufus* (Endangered)

### **3.2 Data Statistical Summary**

Here, statistical summary is generated of `df_observations`. Using `include="all"` covers both the text columns and the numeric column in one table.

In [558]:
# Generate a summary of each column, including categorical and numeric data.
# Transpose the result so each column's statistics appear in a row.
print("Statistical summary of the observations dataset:")
df_observations.describe(include="all").T

Statistical summary of the observations dataset:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
scientific_name,23296,5541,Puma concolor,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
park_name,23296,4,Great Smoky Mountains National Park,5824,NaN,NaN,NaN,NaN,NaN,NaN,NaN
observations,23296.0,NaN,NaN,NaN,142.287904,69.890532,9.0,86.0,124.0,195.0,321.0


- All three columns have 23,296 non-missing values, so `df_observations` has no missing data
- `scientific_name` has 5,541 unique values across 23,296 rows, so each species appears multiple times. The most frequent is *Puma concolor*, with 12 rows
- `park_name` has 4 unique values. The most frequent is Great Smoky Mountains National Park, with 5,824 rows
- `observations` ranges from 9 to 321, with a mean of about 142.3 and a median (50%) of 124. The mean is higher than the median, which suggests the distribution is right-skewed
- The standard deviation of `observations` is about 69.9

The statistical summary for `df_species` is generated here.

In [559]:
# Generate a summary of each column, including categorical and numeric data.
# Transpose the result so each column's statistics appear in a row.
print("Statistical summary of the dataset:")
df_species.describe(include="all").T

Statistical summary of the dataset:


,count,unique,top,freq
category,5824,7,Vascular Plant,4470
scientific_name,5824,5541,Canis lupus,3
common_names,5824,5504,Dicranum Moss,7
conservation_status,191,4,Species of Concern,161


- `df_species` has 5,824 rows. `category`, `scientific_name`, and `common_names` are fully populated, with 5,824 non-missing values each
- `category` has 7 unique values. The most frequent is Vascular Plant, with 4,470 rows
- `scientific_name` has 5,541 unique values across 5,824 rows, so some names appear more than once. The most frequent is *Canis lupus*, with 3 rows
- `common_names` has 5,504 unique values. The most frequent is "Dicranum Moss", with 7 rows
- `conservation_status` has only 191 non-missing values out of 5,824, with 4 unique values. The most frequent is Species of Concern, with 161 rows

The structure, data types, and non-null counts of each dataset is checked with `.info()`.

In [560]:
df_observations.info()

<class 'pandas.DataFrame'>
RangeIndex: 23296 entries, 0 to 23295
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   scientific_name  23296 non-null  str  
 1   park_name        23296 non-null  str  
 2   observations     23296 non-null  int64
dtypes: int64(1), str(2)
memory usage: 546.1 KB


In [561]:
df_species.info()

<class 'pandas.DataFrame'>
RangeIndex: 5824 entries, 0 to 5823
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   category             5824 non-null   str  
 1   scientific_name      5824 non-null   str  
 2   common_names         5824 non-null   str  
 3   conservation_status  191 non-null    str  
dtypes: str(4)
memory usage: 182.1 KB


**`df_observations`**
- 23,296 rows and 3 columns
- All three columns have 23,296 non-null values, so there are no missing values
- `scientific_name` and `park_name` are stored as strings, and `observations` is an integer (`int64`)
- Memory usage is about 546 KB

**`df_species`**
- 5,824 rows and 4 columns
- `category`, `scientific_name`, and `common_names` have 5,824 non-null values each, so they are complete
- `conservation_status` has only 191 non-null values, so it is the only column with missing data
- All four columns are stored as strings
- Memory usage is about 182 KB

## **4. Data Quality Checks**
In this section, checks for missing values, duplicates, unexpected categories, and inconsistent text in key columns.

### **4.1 Consistency of `scientific_name` across datasets**

Both datasets contain a `scientific_name` column. Sets of unique names are compared in each to see how many appear in both and how many appear in only one.

In [562]:
# Convert scientific-name columns to sets to compare unique names.
species_names = set(df_species["scientific_name"])
observation_names = set(df_observations["scientific_name"])

# Find scientific names present in both datasets.
common_names = species_names & observation_names

# Find names present in one dataset but absent from the other.
species_only_names = species_names - observation_names
observations_only_names = observation_names - species_names

# Report the number of unique names in each comparison group.
print(f"Unique species names: {len(species_names)}")
print(f"Unique observation names: {len(observation_names)}")
print(f"Names in both: {len(common_names)}")
print(f"Only in species dataset: {len(species_only_names)}")
print(f"Only in observations dataset: {len(observations_only_names)}")

# List names that do not have a match in the other dataset.
print("\nOnly in species dataset:")
print(sorted(species_only_names))

print("\nOnly in observations dataset:")
print(sorted(observations_only_names))

Unique species names: 5541
Unique observation names: 5541
Names in both: 5541
Only in species dataset: 0
Only in observations dataset: 0

Only in species dataset:
[]

Only in observations dataset:
[]


- Both datasets contain 5,541 unique scientific names
- All 5,541 names appear in both datasets
- No names are found only in the species dataset, and none are found only in the observations dataset (both lists are empty)

The two datasets cover exactly the same set of species.

### **4.2 Checking categorical columns**
Distinct values are listed here of the categorical columns in `df_species` to check for unexpected labels, typos, or inconsistent spelling.

In [563]:
df_species["category"].unique()

<StringArray>
[           'Mammal',              'Bird',           'Reptile',
         'Amphibian',              'Fish',    'Vascular Plant',
 'Nonvascular Plant']
Length: 7, dtype: str

The column has 7 distinct values: Mammal, Bird, Reptile, Amphibian, Fish, Vascular Plant, and Nonvascular Plant. The labels are consistently spelled and capitalized.

In [564]:
df_species["conservation_status"].unique()

<StringArray>
[nan, 'Species of Concern', 'Endangered', 'Threatened', 'In Recovery']
Length: 5, dtype: str

The column has 4 distinct labels plus missing values: Species of Concern, Endangered, Threatened, and In Recovery. The labels are also consistently spelled.

In [565]:
status_counts = df_species["conservation_status"].value_counts(dropna=False)
status_counts

conservation_status
NaN                   5633
Species of Concern     161
Endangered              16
Threatened              10
In Recovery              4
Name: count, dtype: int64

Count each value with `dropna=False` is generated here so missing values are included in the totals.

| conservation_status | Count |
|---|---|
| Missing (NaN) | 5,364 |
| Species of Concern | 151 |
| Endangered | 14 |
| Threatened | 9 |
| In Recovery | 3 |

- The counts add up to 5,541 rows
- Most rows (5,364, about 97%) have no conservation status
- Species of Concern is the most common status, with 151 rows
- Endangered, Threatened, and In Recovery are much rarer, with 14, 9, and 3 rows

### **4.3 Park coverage per species**

The analysis compares parks, so whether each species has records in every park is checked. The check does three things:
1. Collects the distinct park names from `df_observations` and raises an error if there are not exactly 4
2. Builds the set of parks recorded for each `scientific_name`
3. Reports any species whose set of parks differs from the full set of four

In [566]:
# Get the distinct, non-missing parks represented in the observations.
expected_parks = set(df_observations["park_name"].dropna().unique())

# Confirm the dataset contains the expected four parks.
if len(expected_parks) != 4:
    raise ValueError(
        f"Expected 4 parks, but found {len(expected_parks)}: {sorted(expected_parks)}"
    )

# Collect the parks where each scientific name has been observed.
parks_by_species = (
    df_observations.dropna(subset=["scientific_name", "park_name"])
    .groupby("scientific_name")["park_name"]
    .agg(lambda parks: set(parks))
)

# Identify scientific names that are missing observations in one or more parks.
missing_parks = {
    name: sorted(expected_parks - parks)
    for name, parks in parks_by_species.items()
    if parks != expected_parks
}

# Report species with incomplete park coverage, or confirm full coverage.
if missing_parks:
    print(f"{len(missing_parks)} scientific names are missing one or more park entries:")
    for name, parks in missing_parks.items():
        print(f"{name}: missing {parks}")
else:
    print("Every scientific name has entries in all four national parks.")

Every scientific name has entries in all four national parks.


- The dataset contains exactly 4 parks, so the safeguard did not raise an error
- No species is missing from any park, so the check printed that every scientific name has entries in all four national parks

This check confirms which parks each species appears in. It does not count how many rows a species has within each park.

### **4.4 Aggregate Observations by Species**
Each species appears in several rows of `df_observations`. Data is grouped by `scientific_name` and sum `observations`, which gives one row per species with its total observations across all rows.

In [567]:
# Sum observations across all parks to get one total per scientific name.
df_observations_by_species = (
    df_observations
    .groupby("scientific_name", as_index=False)["observations"]
    .sum()
)

# Preview the aggregated results.
df_observations_by_species.head()

,scientific_name,observations
0,Abies bifolia,532
1,Abies concolor,630
2,Abies fraseri,518
3,Abietinella abietina,592
4,Abronia ammophila,523


The result has two columns: `scientific_name` and `observations` (the summed total). The first five species are *Abies bifolia* (532), *Abies concolor* (630), *Abies fraseri* (518), *Abietinella abietina* (592), and *Abronia ammophila* (523).

In [568]:
df_observations_by_species.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
scientific_name,5541,5541,Abies bifolia,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
observations,5541.0,NaN,NaN,NaN,598.220357,137.604129,131.0,544.0,573.0,604.0,1785.0


- There are 5,541 rows, one for each unique `scientific_name`
- Total observations per species range from 131 to 1,785
- The mean is about 598.2 and the median (50%) is 573
- The middle 50% of species fall between 544 and 604 observations
- The maximum (1,785) is far above the 75th percentile (604), so a few species have much higher totals than the rest
- The standard deviation is about 137.6

### **4.5 Checking whether common names identify a single species**
`scientific_name` is the species identifier, whether `common_names` could serve as one can be checked. Distinct scientific names are counted and shared to each common name, and the common names linked to more than one have been kept.

In [569]:
# Find common names associated with more than one scientific name.
common_to_multiple_species = (
    df_species.dropna(subset=["common_names", "scientific_name"])
    .groupby("common_names")["scientific_name"]
    .nunique()
    .loc[lambda counts: counts > 1]
    .sort_values(ascending=False)
)

# Display the shared common names and their scientific-name counts.
print(common_to_multiple_species)

common_names
Brachythecium Moss         7
Dicranum Moss              7
Bryum Moss                 6
Panic Grass                6
Sphagnum                   6
                          ..
Woolly Sedge               2
Yellow Warbler             2
Yellow-Rumped Warbler      2
Yellow-Throated Warbler    2
Zygodon Moss               2
Name: scientific_name, Length: 246, dtype: int64


**Common names shared by multiple species**
- 246 common names are linked to more than one scientific name
- The most widely shared are "Brachythecium Moss" and "Dicranum Moss" (7 scientific names each), followed by "Bryum Moss", "Panic Grass", and "Sphagnum" (6 each)
- The least shared names, such as "Woolly Sedge" and "Yellow-Rumped Warbler", are linked to 2 scientific names

In [570]:
# Display rows for common names linked to multiple scientific names.
# Sort by common name and show the common and scientific name columns.
df_species[
    df_species["common_names"].isin(common_to_multiple_species.index)
].sort_values("common_names")[["common_names", "scientific_name"]]

,common_names,scientific_name
2730,A Moss,Dichodontium pellucidum
2822,A Moss,Cirriphyllum piliferum
2022,"A Sedge, Sedge",Carex normalis
1971,"A Sedge, Sedge",Carex bromoides
1960,"A Sedge, Sedge",Carex annectens
...,...,...
250,Yellow-Rumped Warbler,Dendroica coronata
252,Yellow-Throated Warbler,Dendroica dominica
3206,Yellow-Throated Warbler,Setophaga dominica
2957,Zygodon Moss,Zygodon viridissimus var. rupestris


**Rows behind the shared names**\
Listing the affected rows shows 564 rows across these 246 common names.
- Some are generic labels that cover several species, such as "A Moss", which appears for both *Dichodontium pellucidum* and *Cirriphyllum piliferum*
- Some cells hold more than one name, such as "A Sedge, Sedge"
- Some species with the same common name differ in their scientific name. "Yellow-Throated Warbler" is listed under both *Dendroica dominica* and *Setophaga dominica*, and "Zygodon Moss" under both *Zygodon viridissimus* and *Zygodon viridissimus var. rupestris*

A common name does not point to exactly one scientific name in this dataset.

## **5. Data Cleaning**

### **5.1 Conflicting conservation statuses**
A species should have a single conservation status. For each `scientific_name`, Distinct non-missing statuses were counted and names with more than one were kept.

In [571]:
# Count distinct non-missing conservation statuses for each scientific name.
status_species_counts = (
    df_species.dropna(subset=["conservation_status"])
    .groupby("scientific_name")["conservation_status"]
    .nunique()
)

# Select scientific names associated with more than one status.
names_with_multiple_statuses = status_species_counts[status_species_counts > 1].index

# Collect and display the distinct status records for those species.
multiple_statuses = (
    df_species[df_species["scientific_name"].isin(names_with_multiple_statuses)]
    [["scientific_name", "conservation_status"]]
    .drop_duplicates()
    .sort_values(["scientific_name", "conservation_status"])
)

print(multiple_statuses)

     scientific_name conservation_status
8        Canis lupus          Endangered
3020     Canis lupus         In Recovery


**Result**
Only one scientific name has more than one status: *Canis lupus*, recorded as both Endangered and In Recovery.
This check ignores missing values, so it only finds species with two or more actual statuses.

### **5.2 Removing `common_names` and exact duplicates**
`common_names` column is dropped and then duplicate rows are removed. The column has to go first, because rows that differ only by common name would otherwise not count as duplicates. Index is then reset.

In [572]:
# Remove the non-unique common-name field, drop exact duplicate rows,
# and reset the row index after cleaning.
df_species = (
    df_species
    .drop(columns=["common_names"])
    .drop_duplicates()
    .reset_index(drop=True)
)

In [573]:
df_species.describe(include="all").T

,count,unique,top,freq
category,5543,7,Vascular Plant,4262
scientific_name,5543,5541,Canis lupus,2
conservation_status,180,4,Species of Concern,151


**Result**
- `df_species` now has 5,543 rows, down from 5,824
- `scientific_name` has 5,541 unique values across 5,543 rows
- `conservation_status` has 180 non-missing values, down from 191
- `category` still has 7 unique values, with Vascular Plant the most frequent (4,262 rows)

### **5.3 Checking for remaining repeated names**
There are 5,543 rows but 5,541 unique scientific names, so some names must still appear more than once. Rows are counted again and list is generated of names repeated.

In [589]:
# Count occurrences of each scientific name, including missing values.
name_counts = df_species["scientific_name"].value_counts(dropna=False)

# Keep names that appear in more than one row.
repeated_names = name_counts[name_counts > 1]

# Report dataset size, unique non-missing names, and repeated names.
print(f"Rows: {len(df_species)}")
print(f"Unique non-missing scientific names: {df_species['scientific_name'].nunique()}")
print(f"Scientific names appearing in multiple rows: {len(repeated_names)}")
print(repeated_names)

Rows: 5541
Unique non-missing scientific names: 5541
Scientific names appearing in multiple rows: 0
Series([], Name: count, dtype: int64)


**Result**
Two scientific names appear in more than one row: *Canis lupus* and *Oncorhynchus mykiss*, with 2 rows each.

In [ ]:
# Display all rows for scientific names that occur more than once.
# Sort them by scientific name to make repeated entries easy to inspect.
df_species[
    df_species["scientific_name"].isin(repeated_names.index)
].sort_values("scientific_name")

,category,scientific_name,conservation_status
8,Mammal,Canis lupus,Endangered
3018,Mammal,Canis lupus,In Recovery
560,Fish,Oncorhynchus mykiss,NaN
3247,Fish,Oncorhynchus mykiss,Threatened


**Rows for the repeated names**
Each pair has the same category but different statuses, so exact-duplicate removal did not merge them.

### **5.4 Resolving the repeated names**
For both repeated names, `conservation_status`  is set to missing and drop the duplicates again, so each name keeps a single row. Index is then reset. The 'conservation_status' data for these 2 will be further explored and processed.

In [ ]:
# Species whose conservation-status values will be set to missing.
names_to_clear = ["Canis lupus", "Oncorhynchus mykiss"]

# Replace conservation statuses for the selected species with missing values.
df_species.loc[
    df_species["scientific_name"].isin(names_to_clear),
    "conservation_status"
] = pd.NA

# Remove rows that are now identical and reset the index.
df_species = df_species.drop_duplicates().reset_index(drop=True)

In [577]:
df_species.describe(include="all").T

,count,unique,top,freq
category,5541,7,Vascular Plant,4262
scientific_name,5541,5541,Clethrionomys gapperi gapperi,1
conservation_status,177,4,Species of Concern,151


**Result of resolving the repeated names**
- `df_species` now has 5,541 rows, and `scientific_name` has 5,541 unique values, so each name appears exactly once
- `conservation_status` has 177 non-missing values, down from 180
- `category` is unchanged, with 7 unique values and Vascular Plant the most frequent (4,262 rows)

**Final Preview of both prepared tables and shapes are compared.**

In [578]:
df_observations_by_species.head()

,scientific_name,observations
0,Abies bifolia,532
1,Abies concolor,630
2,Abies fraseri,518
3,Abietinella abietina,592
4,Abronia ammophila,523


In [579]:
df_species.head()

,category,scientific_name,conservation_status
0,Mammal,Clethrionomys gapperi gapperi,NaN
1,Mammal,Bos bison,NaN
2,Mammal,Bos taurus,NaN
3,Mammal,Ovis aries,NaN
4,Mammal,Cervus elaphus,NaN


In [580]:
df_observations_by_species.shape, df_species.shape


((5541, 2), (5541, 3))

**Prepared tables**
- `df_observations_by_species` has two columns, `scientific_name` and `observations`
- `df_species` has three columns, `category`, `scientific_name`, and `conservation_status`
- Both tables have 5,541 rows, matching the 5,541 unique scientific names found earlier
- `scientific_name` is present in both, and each table has one row per name

## **6. Merging of Datasets**
Here, aggregated observations are merged with the cleaned species table on `scientific_name`. A left join keeps every species from the observations table and adds `category` and `conservation_status` to each.

In [581]:
species_observation = df_observations_by_species.merge(df_species, on="scientific_name", how="left")

In [582]:
species_observation

,scientific_name,observations,category,conservation_status
0,Abies bifolia,532,Vascular Plant,NaN
1,Abies concolor,630,Vascular Plant,NaN
2,Abies fraseri,518,Vascular Plant,Species of Concern
3,Abietinella abietina,592,Nonvascular Plant,NaN
4,Abronia ammophila,523,Vascular Plant,Species of Concern
...,...,...,...,...
5536,Zonotrichia leucophrys gambelii,560,Bird,NaN
5537,Zonotrichia leucophrys oriantha,558,Bird,NaN
5538,Zonotrichia querula,616,Bird,NaN
5539,Zygodon viridissimus,600,Nonvascular Plant,NaN


**Merged table preview**
The merged table has 5,541 rows and 4 columns: `scientific_name`, `observations`, `category`, and `conservation_status`. Each row is one species, with its total observations and its metadata.

In [583]:
species_observation.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
scientific_name,5541,5541,Abies bifolia,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
observations,5541.0,NaN,NaN,NaN,598.220357,137.604129,131.0,544.0,573.0,604.0,1785.0
category,5541,7,Vascular Plant,4262,NaN,NaN,NaN,NaN,NaN,NaN,NaN
conservation_status,177,4,Species of Concern,151,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Summary of the merged table**
- `scientific_name` has 5,541 unique values across 5,541 rows, so each species appears once
- `observations` ranges from 131 to 1,785, with a mean of about 598.2 and a median of 573. These match the aggregated table from section 4.4, so the merge did not change them
- `category` has 7 unique values, and Vascular Plant is the most frequent (4,262 species)
- `conservation_status` has 177 non-missing values out of 5,541. Species of Concern is the most frequent, with 151

In [584]:
species_observation.info()

<class 'pandas.DataFrame'>
RangeIndex: 5541 entries, 0 to 5540
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   scientific_name      5541 non-null   str  
 1   observations         5541 non-null   int64
 2   category             5541 non-null   str  
 3   conservation_status  177 non-null    str  
dtypes: int64(1), str(3)
memory usage: 173.3 KB


**Structure of the merged table**
- All 5,541 rows are present, matching the row counts of both input tables
- `scientific_name`, `observations`, and `category` have no missing values
- `conservation_status` has 177 non-null values, which matches the count in the cleaned species table. The left join added no extra missing values
- Data types are as expected: `observations` is an integer and the other three columns are strings

## **7. Exporting Cleaned Data**
Merged table is saved, `species_observation`, to a CSV file in the `data/cleaned/` folder. The code creates the folder if it does not already exist, and `index=False` stops pandas from writing the row index as an extra column.

In [586]:
# Define the output path
cleaned_data_path = Path("../data/cleaned/species_observation_processed.csv")
# Create the directory if it doesn't exist
cleaned_data_path.parent.mkdir(parents=True, exist_ok=True)

# Save the prepared DataFrame to CSV
species_observation.to_csv(cleaned_data_path, index=False)

print(f"Cleaned data saved to: {cleaned_data_path}")

Cleaned data saved to: ..\data\cleaned\species_observation_processed.csv


## **8. Summary**

Two raw datasets were inspected on species observations in four national parks, checked their quality, resolved duplicate and conflicting records, and merged them into one analysis-ready table with one row per species.

### Results at a glance

| Stage | Result |
|---|---|
| Raw observations table | 23,296 rows, 3 columns, no missing values |
| Raw species table | 5,824 rows, 4 columns, 191 non-missing conservation statuses |
| Unique scientific names | 5,541 in both datasets, all matching |
| Park coverage | Every species has entries in all 4 parks |
| Species table after cleaning | 5,541 rows, one per scientific name |
| Final merged table | 5,541 rows x 4 columns |

### Cleaning steps

1. **Checked key consistency.** Both datasets contain the same 5,541 scientific names, so the merge key is reliable.
2. **Reviewed categorical columns.** `category` has 7 consistent labels. `conservation_status` has 4 labels plus missing values.
3. **Aggregated observations.** I summed `observations` for each species, giving 5,541 rows with totals from 131 to 1,785 (mean about 598.2, median 573).
4. **Dropped `common_names`.** 246 common names are shared by more than one scientific name, so the column does not identify a single species.
5. **Removed exact duplicates.** Species rows fell from 5,824 to 5,543.
6. **Resolved repeated names.** *Canis lupus* (Endangered / In Recovery) and *Oncorhynchus mykiss* (missing / Threatened) each appeared twice with different statuses. I set both statuses to missing and kept one row each, leaving 5,541 rows. Data for these two species will be explored and processed further
7. **Merged and exported.** A left join on `scientific_name` produced the final table, saved to `data/cleaned/species_observation_processed.csv`.

### Final dataset

| Column | Description | Missing values |
|---|---|---|
| `scientific_name` | Species identifier (unique per row) | 0 |
| `observations` | Total observations across the dataset | 0 |
| `category` | Species group (7 categories) | 0 |
| `conservation_status` | Protection status (4 labels) | 5,364 |

### Limitations
- **Most statuses are missing.** About 97% of species (5,364 of 5,541) have no conservation status. I kept these as missing, and they may mean the species has no protected status.
- **Status conflicts check ignores blanks.** The first conflict check dropped missing values, so a species with one blank and one real status (like *Oncorhynchus mykiss*) was found only by the later repeated-name check.

### Next steps
- Explore observations by category, conservation status, and species
- Revisit observations for species *Canis lupus* and *Oncorhynchus mykiss* for further explorationa and processing.